# Hier werden die Daten in die jeweilige Form gebracht, die zur Analyse (in R) gebraucht wird

In [1]:
import pandas as pd
import os
import numpy as np
from scipy import stats
import seaborn as sns
import matplotlib.pyplot as plt



os.chdir(r"C:\Users\hundh\Desktop\Python\Daten")
pd.set_option('display.max_colwidth', None)
df = pd.read_csv("kandis_vercodet.csv")
df.shape[0]

df_später = df
df["date"] = pd.to_datetime(df["date"])

## Erster Teil: Die Daten mit den Posts als Reihen werden um kleinere zusätzliche Infos ergänzt

In [2]:
# in df_count wird gezählt, wie viele posts jeder Kandidierende abgesetzt hat
df_count = df.groupby("id_scraping", as_index=False).size().copy(deep=True)
df_count = df_count.loc[:,["id_scraping","size"]]
df_count.rename(columns={"size":"num_posts"}, inplace=True)

df_count.shape[0]

1815

In [3]:
# Um weitere Infos über die Kandidierenden zu erhalten, wird der amtliche Datensatz hinzugenommen
df_kandis = pd.read_csv("übersicht_kandis.csv", sep=';')

# Der Datensatz wird bereinigt; insbesondere werden die letzten 10 Zeilen gelöscht. Diese enthalten keine Kandidierenden.
df_kandis.rename(columns={"ID":"id_scraping"}, inplace=True)
df_kandis.drop(df_kandis.tail(10).index, inplace=True)
df_kandis["id_scraping"] = df_kandis["id_scraping"].astype(int)

df_kandis.shape[0] #Anzahl der Kandidierenden (Größer, weil in Daten_Cleaning einige Kandidierende gelöscht werden)

4506

In [4]:
# Die Anzahl der Posts eines Kandidierenden (num_posts aus df_count) und die Variable, die angibt, ob ein Kandidierender als Listenkandidierender oder nicht antritt (Kennzeichen aus df_kandis), wird zum originalen Datensatz hinzugefügt
df = pd.merge(df, df_count, on="id_scraping")
df = pd.merge(df, df_kandis, on="id_scraping")
df.rename(columns={"size":"num_posts"}, inplace=True)

In [5]:
#Namen werden vereinheitlicht
df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)

C:\Users\hundh\AppData\Local\Temp\ipykernel_28832\1165275403.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)


In [6]:
# Es werden nicht alle Informationen mitgenommen
df = df[["id","id_scraping","author","text","num_followers","Geschlecht","Geburtsjahr","GruppennameKurz","acc_type","date","period","Ost_West","D_0","D_1","D_2","D_3","D_4","D_5","D_6","D_7","num_posts","Kennzeichen"]]
df.shape[0]

83581

In [7]:
df.to_csv("kandis_uncombined.csv") # Dieser Datensatz wird für den ersten Teil der Analyse (Also Kapitel 3.1, Tab. 1&2 und 3.2) verwendet

## Zweiter Teil

In [8]:
df = df_später

#Wenn die conditions gelten, wurde der Post nach Veröffentlichung des Wahlprogramms der jeweiligen Partei abgesetzt

condition = [(df["date"].ge("2025-02-09") & df["party"].eq("FDP")) |
             (df["date"].ge("2024-12-17") & df["party"].eq("CDU")) |
             (df["date"].ge("2024-12-17") & df["party"].eq("CSU")) |
             (df["date"].ge("2025-01-11") & df["party"].eq("SPD")) |
             (df["date"].ge("2025-01-12") & df["party"].eq("BSW")) |
             (df["date"].ge("2025-01-18") & df["party"].eq("Die Linke")) |
             (df["date"].ge("2025-01-26") & df["party"].eq("GRÜNE")) |
             (df["date"].ge("2025-02-03") & df["party"].eq("AfD"))]

choice = [1]

df["nach_wahlprogramm"] = np.select(condition,choice, default=0)

df_nach_wahlprogramm = df[df["nach_wahlprogramm"] == 1]
df_vor_wahlprogramm = df[df["nach_wahlprogramm"] == 0]

#### Es folgt drei mal das gleiche Vorgehen. Einmal für den kompletten Datensatz, einmal für den Datensatz mit Posts, die nach der Veröffentlichung der Wahlprogramme abgesetzt wurden und einmal für den Datensatz mit Posts, die vor der Veröffentlichung der Wahlprogramme abgesetzt wurden. 
##### Für die Arbeit ist eigentlich nur der Datensatz vor und nach relevant aber die Anzahl der insgesamten Fälle soll ermittelt werden. Deshalb wird das Vorgehen für alle drei Zeiträume durchgeführt.

In [9]:
# in df_count wird gezählt, wie viele posts jeder Kandidierende abgesetzt hat
df_count = df.groupby("id_scraping", as_index=False).size().copy(deep=True)
df_count = df_count.loc[:,["id_scraping","size"]]
df_count.rename(columns={"size":"num_posts"}, inplace=True)

# in df_issue_count wird gezählt, wie viele issue-posts jeder Kandidierende abgesetzt hat
df_issue_count = df[df["D_0"] != 1] #Damit D_0 nicht die Issue-Anteile und die Anzahl der Posts verzerrt, wird es hier gelöscht!
df_issue_count = df_issue_count.groupby("id_scraping", as_index=False).size()
df_issue_count = df_issue_count.loc[:,["id_scraping","size"]]
df_issue_count.rename(columns={"size":"num_issue_posts"}, inplace=True)

# Datensätze werden kombiniert und dann eine Variable für den Anteil an Issue-Posts unter den gesamten Posts für jede/n Kandidierende/n berechnet.
# Diese neue Variable (share_issue) gibt an, ob Kandidierende im Wahlkampf eher auf Themenbetonung (share_issue groß) oder auf andere Strategien gesetzt haben
df_count = pd.merge(df_count, df_issue_count, on="id_scraping")
df_count["share_issue"] = df_count["num_issue_posts"] /df_count["num_posts"]

df_count_later = df_count #Diese Daten werden für den zweiten Teil gebraucht (siehe unten). Da sich der zweite Teil auf alle Posts bezieht, muss hier zwischengespeichert werden

In [10]:
# in df wird gezählt, wie viele posts die jeweiligen Themen ansprechen und dann wird mit df_count gemerged
df = df.groupby(["id_scraping","author","author_fullname","sex", "num_followers","Ost_West"], as_index=False)[["D_1","D_2","D_3","D_4","D_5","D_6","D_7"]].sum()
df = pd.merge(df, df_count, on="id_scraping")

In [11]:
# Hier wird ausgerechnet, wie groß der Anteil jedes Issues an den gesamten erwähnten Issues eine Kandidierenden ist (also z.B. in 10% aller issuebezogenen Posts wird das Thema Außenpolitik (D_1) angesprochen)
df["sum"] = df["D_1"] + df["D_2"] + df["D_3"] + df["D_4"] + df["D_5"] + df["D_6"] + df["D_7"]

df = df[df["sum"] != 0]  #Summe ist 0 wenn kein Post ein Issue anspricht. Diese Kandidierenden sind nicht brauchbar

df["D_1"] = df["D_1"]/ df["sum"]
df["D_2"] = df["D_2"]/ df["sum"]
df["D_3"] = df["D_3"]/ df["sum"]
df["D_4"] = df["D_4"]/ df["sum"]
df["D_5"] = df["D_5"]/ df["sum"]
df["D_6"] = df["D_6"]/ df["sum"]
df["D_7"] = df["D_7"]/ df["sum"]

In [12]:
# Um weitere Infos über die Kandidierenden zu erhalten, wird der amtliche Datensatz hinzugenommen
df_kandis = pd.read_csv("übersicht_kandis.csv", sep=';')

# Der Datensatz wird bereinigt; insbesondere werden die letzten 10 Zeilen gelöscht. Diese enthalten keine Kandidierenden.
df_kandis.rename(columns={"ID":"id_scraping"}, inplace=True)
df_kandis.drop(df_kandis.tail(10).index, inplace=True)
df_kandis["id_scraping"] = df_kandis["id_scraping"].astype(int)
df_kandis_later = df_kandis #Diese Daten werden für den zweiten Teil gebraucht (siehe unten). Da sich der zweite Teil auf alle Posts bezieht, muss hier zwischengespeichert werden

In [13]:
# Dann wird er mit dem Hauptdatensatz gemerged und die Parteinamen angepasst
df = pd.merge(df, df_kandis, on=["id_scraping"])
df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)

C:\Users\hundh\AppData\Local\Temp\ipykernel_23568\2318335984.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)


In [14]:
# Der Manifesto-Project Datensatz wird geladen. Jede Reihe stellt eine Partei da. Um die Anteile der einzelnen Issues für unsere Partei zu erhalten, filtern wir nach Land und Wahldatum
df_parties = pd.read_csv("MPD_data.csv")
df_parties = df_parties[df_parties["country"] == 41]
df_parties = df_parties[df_parties["date"] == 202502]
df_parties = df_parties[df_parties["partyabbrev"] != "SSW"]

C:\Users\hundh\AppData\Local\Temp\ipykernel_23568\2357188998.py:2: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_parties = pd.read_csv("MPD_data.csv")


In [15]:
## Die Subkategorien werden entsprechend unserem Codebook zu Hauptkategorien zusammengefasst.

# Domain 1: External Relations 
df_parties["D_1"] = df_parties["per101"] + df_parties["per102"] +df_parties["per103"] +df_parties["per104"] +df_parties["per105"] +df_parties["per106"] +df_parties["per107"] +df_parties["per108"] +df_parties["per109"] +df_parties["per110"]
# Domain 2: Freedom, Democracy and Political System
df_parties["D_2"] = df_parties["per201"] + df_parties["per202"] + df_parties["per203"] + df_parties["per204"] + df_parties["per301"] + df_parties["per302"] + df_parties["per303"] + df_parties["per304"] 
# Domain 3: Economy, Finance and Bureaucracy
df_parties["D_3"] = df_parties["per401"] + df_parties["per402"] + df_parties["per403"] + df_parties["per404"] + df_parties["per405"] + df_parties["per406"] + df_parties["per407"] + df_parties["per408"] + df_parties["per409"] + df_parties["per410"] + df_parties["per411"] + df_parties["per412"] + df_parties["per413"] + df_parties["per414"] + df_parties["per415"] + df_parties["per701"]+ df_parties["per702"]+ df_parties["per703"]+ df_parties["per704"]
#Domain 4: Welfare and Quality of Life
df_parties["D_4"] = df_parties["per503"] + df_parties["per504"] + df_parties["per505"] + df_parties["per506"] + df_parties["per507"] + df_parties["per705"] + df_parties["per706"]
# Domain 5:Environment 
df_parties["D_5"] = df_parties["per501"] + df_parties["per416"]
# Domain 6: Fabric of Society
df_parties["D_6"] = df_parties["per502"] + df_parties["per601_1"] + df_parties["per602_1"] + df_parties["per603"] + df_parties["per604"] + df_parties["per605"] + df_parties["per606"]
# Domain 7: Migration and Multiculturalism
df_parties["D_7"] = df_parties["per607"] + df_parties["per608"] + df_parties["per601_2"] + df_parties["per602_2"]

# Summe
df_parties["sum"] = df_parties["D_1"] + df_parties["D_2"] + df_parties["D_3"] + df_parties["D_4"] + df_parties["D_5"] + df_parties["D_6"] + df_parties["D_7"]

df_parties["D_1"] = df_parties["D_1"]/ df_parties["sum"]
df_parties["D_2"] = df_parties["D_2"]/ df_parties["sum"]
df_parties["D_3"] = df_parties["D_3"]/ df_parties["sum"]
df_parties["D_4"] = df_parties["D_4"]/ df_parties["sum"]
df_parties["D_5"] = df_parties["D_5"]/ df_parties["sum"]
df_parties["D_6"] = df_parties["D_6"]/ df_parties["sum"]
df_parties["D_7"] = df_parties["D_7"]/ df_parties["sum"]

In [16]:
df_parties[["partyabbrev","D_1","D_2","D_3","D_4","D_5","D_6","D_7"]]

,partyabbrev,D_1,D_2,D_3,D_4,D_5,D_6,D_7
2076,90/Greens,0.103388,0.156578,0.217353,0.239438,0.144807,0.108818,0.029619
2077,LINKE,0.090471,0.095819,0.310601,0.319514,0.117636,0.050361,0.015598
2078,SPD,0.134334,0.111226,0.281478,0.269199,0.060038,0.122555,0.021169
2079,FDP,0.093744,0.205958,0.343039,0.198158,0.030538,0.085235,0.043327
2080,CDU/CSU,0.129770,0.119119,0.341341,0.152883,0.056887,0.152437,0.047563
2081,BSW,0.135857,0.181106,0.296244,0.244042,0.036840,0.074438,0.031473
2083,AfD,0.108447,0.179288,0.276290,0.129700,0.040875,0.167852,0.097548


In [18]:
# Statt 7 Reihen für die Anteile der 7 Themen am Wahlprogramm wollen wir eine Reihe mit den Themen als Argumente und eine Reihe mit den Anteilen als Argumente (wie bei pivot_longer in R)

df_parties["partyabbrev"].replace({"90/Greens":"GRÜNE"}, inplace=True)
df_parties.rename(columns={"partyabbrev":"GruppennameKurz"}, inplace=True)
df_parties = pd.melt(df_parties, id_vars="GruppennameKurz", value_vars=["D_1","D_2","D_3","D_4","D_5","D_6","D_7"])
df_parties.rename(columns={"value":"Anteil_Partei"}, inplace=True)

C:\Users\hundh\AppData\Local\Temp\ipykernel_23568\2894087739.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_parties["partyabbrev"].replace({"90/Greens":"GRÜNE"}, inplace=True)


In [19]:
# Gleiches für den Hauptdatensatz
df = pd.melt(df, id_vars=["id_scraping","author","author_fullname","sex", "num_followers","Ost_West","GruppennameKurz","num_posts","num_issue_posts","share_issue","Kennzeichen","GebietLandAbk","Listenplatz","Geburtsjahr"], value_vars=["D_1","D_2","D_3","D_4","D_5","D_6","D_7"])
df.rename(columns={"value":"Anteil_Kand"}, inplace=True)

In [20]:
# Merge und als csv gespeichert
df = pd.merge(df, df_parties, on=["GruppennameKurz","variable"])
df.to_csv("kandis_party_merge.csv")

### Nach Veröffentlichung der Wahlprogramme

In [21]:
df = df_nach_wahlprogramm

In [22]:
# in df_count wird gezählt, wie viele posts jeder Kandidierende abgesetzt hat
df_count = df.groupby("id_scraping", as_index=False).size().copy(deep=True)
df_count = df_count.loc[:,["id_scraping","size"]]
df_count.rename(columns={"size":"num_posts"}, inplace=True)

# in df_issue_count wird gezählt, wie viele issue-posts jeder Kandidierende abgesetzt hat
df_issue_count = df[df["D_0"] != 1] #Damit D_0 nicht die Issue-Anteile und die Anzahl der Posts verzerrt, wird es hier gelöscht!
df_issue_count = df_issue_count.groupby("id_scraping", as_index=False).size()
df_issue_count = df_issue_count.loc[:,["id_scraping","size"]]
df_issue_count.rename(columns={"size":"num_issue_posts"}, inplace=True)

# Datensätze werden kombiniert und dann eine Variable für den Anteil an Issue-Posts unter den gesamten Posts für jede/n Kandidierende/n berechnet.
# Diese neue Variable (share_issue) gibt an, ob Kandidierende im Wahlkampf eher auf Themenbetonung (share_issue groß) oder auf andere Strategien gesetzt haben
df_count = pd.merge(df_count, df_issue_count, on="id_scraping")
df_count["share_issue"] = df_count["num_issue_posts"] /df_count["num_posts"]

In [23]:
# in df wird gezählt, wie viele posts die jeweiligen Themen ansprechen und dann wird mit df_count gemerged
df = df.groupby(["id_scraping","author","author_fullname","sex", "num_followers","Ost_West"], as_index=False)[["D_1","D_2","D_3","D_4","D_5","D_6","D_7"]].sum()
df = pd.merge(df, df_count, on="id_scraping")

In [24]:
# Hier wird ausgerechnet, wie groß der Anteil jedes Issues an den gesamten erwähnten Issues eine Kandidierenden ist (also z.B. in 10% aller issuebezogenen Posts wird das Thema Außenpolitik (D_1) angesprochen)
df["sum"] = df["D_1"] + df["D_2"] + df["D_3"] + df["D_4"] + df["D_5"] + df["D_6"] + df["D_7"]

df = df[df["sum"] != 0]  #Summe ist 0 wenn kein Post ein Issue anspricht. Diese Kandidierenden sind nicht brauchbar

df["D_1"] = df["D_1"]/ df["sum"]
df["D_2"] = df["D_2"]/ df["sum"]
df["D_3"] = df["D_3"]/ df["sum"]
df["D_4"] = df["D_4"]/ df["sum"]
df["D_5"] = df["D_5"]/ df["sum"]
df["D_6"] = df["D_6"]/ df["sum"]
df["D_7"] = df["D_7"]/ df["sum"]

In [25]:
# Dann wird er mit dem Hauptdatensatz gemerged und die Parteinamen angepasst
df = pd.merge(df, df_kandis, on=["id_scraping"])
df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)

C:\Users\hundh\AppData\Local\Temp\ipykernel_23568\2318335984.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)


In [26]:
# Entsprechend der Zeile davor für den Hauptdatensatz
df = pd.melt(df, id_vars=["id_scraping","author","author_fullname","sex", "num_followers","Ost_West","GruppennameKurz","num_posts","num_issue_posts","share_issue","Kennzeichen","GebietLandAbk","Listenplatz","Geburtsjahr"], value_vars=["D_1","D_2","D_3","D_4","D_5","D_6","D_7"])
df.rename(columns={"value":"Anteil_Kand"}, inplace=True)

df = pd.merge(df, df_parties, on=["GruppennameKurz","variable"])
df.to_csv("kandis_party_merge_nach.csv")

### Vor Veröffentlichung der Wahlprogramme

In [27]:
df = df_vor_wahlprogramm

In [28]:
# in df_count wird gezählt, wie viele posts jeder Kandidierende abgesetzt hat
df_count = df.groupby("id_scraping", as_index=False).size().copy(deep=True)
df_count = df_count.loc[:,["id_scraping","size"]]
df_count.rename(columns={"size":"num_posts"}, inplace=True)

# in df_issue_count wird gezählt, wie viele issue-posts jeder Kandidierende abgesetzt hat
df_issue_count = df[df["D_0"] != 1] #Damit D_0 nicht die Issue-Anteile und die Anzahl der Posts verzerrt, wird es hier gelöscht!
df_issue_count = df_issue_count.groupby("id_scraping", as_index=False).size()
df_issue_count = df_issue_count.loc[:,["id_scraping","size"]]
df_issue_count.rename(columns={"size":"num_issue_posts"}, inplace=True)

# Datensätze werden kombiniert und dann eine Variable für den Anteil an Issue-Posts unter den gesamten Posts für jede/n Kandidierende/n berechnet.
# Diese neue Variable (share_issue) gibt an, ob Kandidierende im Wahlkampf eher auf Themenbetonung (share_issue groß) oder auf andere Strategien gesetzt haben
df_count = pd.merge(df_count, df_issue_count, on="id_scraping")
df_count["share_issue"] = df_count["num_issue_posts"] /df_count["num_posts"]

In [29]:
# in df wird gezählt, wie viele posts die jeweiligen Themen ansprechen und dann wird mit df_count gemerged
df = df.groupby(["id_scraping","author","author_fullname","sex", "num_followers","Ost_West"], as_index=False)[["D_1","D_2","D_3","D_4","D_5","D_6","D_7"]].sum()
df = pd.merge(df, df_count, on="id_scraping")

In [30]:
# Hier wird ausgerechnet, wie groß der Anteil jedes Issues an den gesamten erwähnten Issues eine Kandidierenden ist (also z.B. in 10% aller issuebezogenen Posts wird das Thema Außenpolitik (D_1) angesprochen)
df["sum"] = df["D_1"] + df["D_2"] + df["D_3"] + df["D_4"] + df["D_5"] + df["D_6"] + df["D_7"]

df = df[df["sum"] != 0]  #Summe ist 0 wenn kein Post ein Issue anspricht. Diese Kandidierenden sind nicht brauchbar

df["D_1"] = df["D_1"]/ df["sum"]
df["D_2"] = df["D_2"]/ df["sum"]
df["D_3"] = df["D_3"]/ df["sum"]
df["D_4"] = df["D_4"]/ df["sum"]
df["D_5"] = df["D_5"]/ df["sum"]
df["D_6"] = df["D_6"]/ df["sum"]
df["D_7"] = df["D_7"]/ df["sum"]

In [31]:
# Dann wird er mit dem Hauptdatensatz gemerged und die Parteinamen angepasst
df = pd.merge(df, df_kandis, on=["id_scraping"])
df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)

C:\Users\hundh\AppData\Local\Temp\ipykernel_23568\2318335984.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["GruppennameKurz"].replace({"CSU":"CDU/CSU","CDU":"CDU/CSU","GRÜNE/B 90":"GRÜNE","Die Linke":"LINKE"}, inplace=True)


In [32]:
# Entsprechend der Zeile davor für den Hauptdatensatz
df = pd.melt(df, id_vars=["id_scraping","author","author_fullname","sex", "num_followers","Ost_West","GruppennameKurz","num_posts","num_issue_posts","share_issue","Kennzeichen","GebietLandAbk","Listenplatz","Geburtsjahr"], value_vars=["D_1","D_2","D_3","D_4","D_5","D_6","D_7"])
df.rename(columns={"value":"Anteil_Kand"}, inplace=True)

df = pd.merge(df, df_parties, on=["GruppennameKurz","variable"])
df.to_csv("kandis_party_merge_vor.csv")